# Actividad 4 — Autocorrelación Espacial (Elecciones Chile 2025)

**Curso:** Análisis de Datos Espaciales · USM
**Duración:** ~60 minutos
**Datos:** Resultados segunda vuelta presidencial 2025, 52 comunas de la Región Metropolitana
**Variable principal:** `Pct_Kast` (% de votos Kast por comuna)

En esta actividad aplicarás los cuatro estadísticos de **autocorrelación espacial global** vistos en la clase 4 al voto presidencial chileno de 2025 en la RM. En cada ejercicio se te pide **calcular** el estadístico y **interpretar** qué dice sobre el patrón espacial del voto.

Al final de cada ejercicio hay un espacio `**Respuesta:**` — escribe ahí tu interpretación con tus propias palabras, no basta con pegar el número.

---


## Setup — Carga de datos

Las siguientes celdas cargan los datos y construyen la cartografía. **Ejecútalas sin modificarlas** antes de empezar los ejercicios.

In [ ]:
# Gráficos
import matplotlib.pyplot as plt
import seaborn
import splot
from splot.esda import plot_moran
import contextily

# Análisis
import geopandas
import pandas
import esda
from libpysal import weights
from numpy.random import seed

In [ ]:
# Cargar resultados electorales de la segunda vuelta 2025 (RM)
elec = pandas.read_csv("datos/external/elecciones2025/resultados_2v_rm.csv")
elec.head()

In [ ]:
# Cargar geometrías de comunas de la RM (Censo 2017)
comunas = geopandas.read_file(
    "datos/external/censo2017/R13/COMUNA_C17.shp"
)
comunas.head()

In [ ]:
# Unir datos electorales con geometrías por nombre de comuna
db = comunas.merge(elec, on="NOM_COMUNA", how="inner")

# Reproyectar a EPSG:3857 (Web Mercator, para usar basemaps)
db = db.to_crs(epsg=3857)

print(f"Comunas con datos: {len(db)} de {len(comunas)}")
db[["NOM_COMUNA", "Pct_Kast", "Pct_Jara", "Ganador"]].head(10)

---
## Ejercicio 1 — Mapa coroplético y matriz de pesos (10 min)

**1.1** Construye un **mapa coroplético** del `Pct_Kast` con esquema de clasificación `Quantiles` y 5 clases, usando una paleta divergente (ej: `coolwarm` o `RdBu_r`) o una secuencial. Agrega título.


In [ ]:
# Tu código aquí


**1.2** Construye una **matriz de pesos espaciales** por k-vecinos más cercanos con `k=8` usando `weights.KNN.from_dataframe` con `ids="NOM_COMUNA"`. Estandarízala por filas (`w.transform = "R"`).


In [ ]:
# Tu código aquí


**1.3** Observando el mapa: ¿detectas a simple vista algún patrón espacial en el voto Kast? ¿Qué zonas de la RM parecen agrupar valores altos y cuáles valores bajos?

**Respuesta:** *(escribe tu interpretación aquí)*


---
## Ejercicio 2 — Rezago espacial (10 min)

**2.1** Calcula el **rezago espacial** de `Pct_Kast` usando `weights.lag_spatial(w, ...)` y guárdalo como columna `Pct_Kast_lag`.


In [ ]:
# Tu código aquí


**2.2** Elige dos comunas con voto Kast contrastante (una alta, una baja) y compara su valor observado con el rezago espacial de sus vecinas. Imprime los resultados.


In [ ]:
# Tu código aquí


**2.3** Dibuja dos mapas lado a lado: el `Pct_Kast` original y el `Pct_Kast_lag`. Usa la misma escala de color.


In [ ]:
# Tu código aquí


**2.4** ¿Qué efecto produce el rezago espacial sobre las diferencias del mapa original? ¿En qué zonas las comunas 'se parecen más' a sus vecinas y en qué zonas hay discrepancias?

**Respuesta:** *(escribe tu interpretación aquí)*


---
## Ejercicio 3 — Caso binario: Join Counts (10 min)

**3.1** Crea una variable binaria `Kast_bin` que valga `1` si `Pct_Kast > 50` y `0` en otro caso. Dibuja el mapa binario.


In [ ]:
# Tu código aquí


**3.2** Construye una **matriz de pesos binaria** (sin estandarizar) con el mismo criterio KNN(k=8) y calcula el estadístico **Join Counts** con `esda.join_counts.Join_Counts(Kast_bin, w_bin)`. Imprime conteos observados (BB, WW, BW) y pseudo p-valores.


In [ ]:
# Tu código aquí


**3.3** ¿Qué indica el resultado? ¿Hay más uniones BB o WW de lo esperado bajo aleatoriedad? ¿Puedes concluir si existe clustering binario del voto Kast?

**Respuesta:** *(escribe tu interpretación aquí)*


---
## Ejercicio 4 — Caso continuo: I de Moran y C de Geary (15 min)

**4.1** Centra `Pct_Kast` restándole su media y guárdalo como `Pct_Kast_std`. Calcula su rezago espacial y guárdalo como `Pct_Kast_lag_std`.


In [ ]:
# Tu código aquí


**4.2** Dibuja el **Gráfico de Moran** usando `seaborn.regplot` (eje X: `Pct_Kast_std`, eje Y: `Pct_Kast_lag_std`). Agrega líneas horizontales y verticales en 0 para marcar los cuatro cuadrantes.


In [ ]:
# Tu código aquí


**4.3** Calcula formalmente el **I de Moran** con `esda.moran.Moran` e imprime su valor y pseudo p-valor. Luego usa `splot.esda.plot_moran` para ver la distribución de permutaciones.


In [ ]:
# Tu código aquí


**4.4** Calcula el **C de Geary** con `esda.geary.Geary` e imprime su valor y pseudo p-valor.


In [ ]:
# Tu código aquí


**4.5** Interpreta y compara:

- ¿Qué signo tiene el I de Moran? ¿Confirma lo que veías en el mapa?
- ¿El C de Geary es consistente con el I de Moran? Recuerda que la interpretación es inversa: valores < 1 indican clustering.
- ¿Qué comuna (o pocas) destacan en el gráfico de Moran como **outliers espaciales** (cuadrantes alto-bajo o bajo-alto)?

**Respuesta:** *(escribe tu interpretación aquí)*


---
## Ejercicio 5 — G de Getis-Ord y hot spots (15 min)

**5.1** Reproyecta `db` al CRS UTM 19S (`epsg=32719`) para trabajar con distancias en metros. Calcula los centroides y el umbral mínimo de distancia con `weights.util.min_threshold_distance`.


In [ ]:
# Tu código aquí


**5.2** Construye la matriz `w_db` con `weights.DistanceBand.from_dataframe` y calcula el **G de Getis-Ord global** con `esda.getisord.G(db["Pct_Kast"], w_db)`. Imprime G y pseudo p-valor. Interprete.


In [ ]:
# Tu código aquí


In [ ]:
# Tu código aquí
